# Deep Learning and GenAI on the Same Data

Notebook for **Chapter 7** of the *ways* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ways/ch07.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
# Setup: the files used in this chapter are downloaded next to the notebook.
import os, urllib.request
os.makedirs("data", exist_ok=True)
for f in ["customers.csv", "orders.csv", "products.csv", "shopkart.db"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://dadipavan.github.io/Learning/study/ways/data/" + f, "data/" + f)


## 7.1 When does deep learning help on a table?

A neural network is a stack of simple functions: a weighted sum, a bend (a non-linear activation such as tanh), another weighted sum, a final sigmoid for a probability. Training adjusts the weights to reduce a loss by gradient descent. Neural networks shine when the input is images, audio, text or very large data, where they learn their own features. On *small tables*, carefully engineered features and tree or linear models usually match them. We test that claim instead of repeating it.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
tr, te = d[d.order_date < "2024-10-01"], d[d.order_date >= "2024-10-01"]
F = mlprep.CAT + mlprep.NUM
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
Xtr, Xte = pre.fit_transform(tr[F]), pre.transform(te[F])
ytr, yte = tr.late.values, te.late.values

rng = np.random.default_rng(0)
H = 16
W1, b1 = rng.normal(0, 0.3, (Xtr.shape[1], H)), np.zeros(H)
W2, b2 = rng.normal(0, 0.3, H), -1.8
sigmoid = lambda z: 1 / (1 + np.exp(-z))
lr, losses = 0.5, []
for epoch in range(400):
    A = np.tanh(Xtr @ W1 + b1)
    P = sigmoid(A @ W2 + b2)
    losses.append(-np.mean(ytr * np.log(P + 1e-9) + (1 - ytr) * np.log(1 - P + 1e-9)))
    dz = (P - ytr) / len(ytr)
    gW2, gb2 = A.T @ dz, dz.sum()
    dA = np.outer(dz, W2) * (1 - A ** 2)
    gW1, gb1 = Xtr.T @ dA, dA.sum(0)
    W1 -= lr * gW1; b1 -= lr * gb1; W2 -= lr * gW2; b2 -= lr * gb2
print("loss at epochs 0, 100, 399:", [round(float(losses[i]), 4) for i in (0, 100, 399)])

P_np = sigmoid(np.tanh(Xte @ W1 + b1) @ W2 + b2)
mlp = MLPClassifier((16,), max_iter=500, random_state=0).fit(Xtr, ytr)
lr_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("numpy network AUC   :", round(roc_auc_score(yte, P_np), 4))
print("sklearn MLP AUC     :", round(roc_auc_score(yte, mlp.predict_proba(Xte)[:, 1]), 4))
print("logistic regression :", round(roc_auc_score(yte, lr_model.predict_proba(Xte)[:, 1]), 4))

loss at epochs 0, 100, 399: [0.4081, 0.31, 0.3043]
numpy network AUC   : 0.8428
sklearn MLP AUC     : 0.8351
logistic regression : 0.8438


## 7.2 Text as numbers: TF-IDF by hand and in scikit-learn

A model cannot read words. The oldest useful trick turns a document into a vector of word counts, then reweights so that common words matter less. **TF-IDF** (term frequency times inverse document frequency) gives a word a high weight if it appears often in *this* document and in few others. With N documents and a word appearing in df of them: tf-idf = tf × (ln((1 + N)/(1 + df)) + 1) in scikit-learn's smoothed version, then each document vector is scaled to length 1.

In [ ]:
DATA = "data/"
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["fresh rice rice", "fresh tea", "premium tea"]
v = TfidfVectorizer()
X = v.fit_transform(docs).toarray()
print(v.get_feature_names_out())
print(np.round(X, 4))

idf = lambda df: np.log((1 + 3) / (1 + df)) + 1
raw = np.array([1 * idf(2), 2 * idf(1)])
print("by hand d1:", np.round(raw / np.linalg.norm(raw), 4))

['fresh' 'premium' 'rice' 'tea']
[[0.3554 0.     0.9347 0.    ]
 [0.7071 0.     0.     0.7071]
 [0.     0.796  0.     0.6053]]
by hand d1: [0.3554 0.9347]


In [ ]:
DATA = "data/"
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

p = pd.read_csv(DATA + "products.csv")
V = TfidfVectorizer()
T = V.fit_transform(p.description)
print("vocabulary size:", len(V.vocabulary_), "| matrix shape:", T.shape)
i = p.index[p["product"] == "Earbuds"][0]
print("description:", p.description[i])
sim = cosine_similarity(T[i], T).ravel()
for j in np.argsort(-sim)[1:4]:
    print(f"  {sim[j]:.3f}  {p['product'][j]:14s} {p.description[j]}")

svd = TruncatedSVD(5, random_state=0).fit(T)
print("variance explained by 5 SVD dimensions:", round(svd.explained_variance_ratio_.sum(), 3))

vocabulary size: 67 | matrix shape: (32, 67)
description: classic earbuds for electronics shoppers, everyday and gentle, good value
  0.478  USB Cable      everyday usb cable for electronics shoppers, gentle and premium, good value
  0.409  Mouse          gentle mouse for electronics shoppers, light and fresh, good value
  0.360  Bedsheet       everyday bedsheet for home shoppers, fresh and classic, good value
variance explained by 5 SVD dimensions: 0.317


## 7.3 A question-answering search you can run: retrieval first

A RAG system answers a question in two steps: **retrieve** the most relevant passages from your own documents, then **generate** an answer with an LLM that is told to use only those passages. The retrieval half is classical information retrieval and runs here. Its quality caps the quality of the whole system: if the right passage is not retrieved, no LLM can answer from it.

In [ ]:
DATA = "data/"
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

p = pd.read_csv(DATA + "products.csv")
V = TfidfVectorizer(ngram_range=(1, 2))
T = V.fit_transform(p["product"].str.lower() + " " + p.description)

def search(q, k=3):
    s = cosine_similarity(V.transform([q]), T).ravel()
    idx = np.argsort(-s)[:k]
    return [(p["product"][i], p.category[i], round(float(s[i]), 3)) for i in idx]

for q in ["something to keep my phone charged", "olive oil for cooking", "durable sneakers", "gentle face wash"]:
    print(q, "->", search(q))

# evaluation: query = product name; is the right product ranked first?
hit1 = np.mean([search(name.lower(), 1)[0][0] == name for name in p["product"]])
# vaguer query: only category words; is a product of that category in the top 3?
cat_hit = np.mean([any(c == cat for _, c, _ in search(f"{cat.lower()} shoppers everyday", 3)) for cat in p.category.unique()])
print("top-1 accuracy for exact product-name queries:", round(hit1, 3))
print("top-3 category hit rate for vague category queries:", round(cat_hit, 3))

something to keep my phone charged -> [('Rice 5kg', 'Grocery', 0.0), ('Lip Balm', 'Beauty', 0.0), ('Sunscreen', 'Beauty', 0.0)]
olive oil for cooking -> [('Olive Oil 1L', 'Grocery', 0.571), ('Hair Oil', 'Beauty', 0.303), ('T-Shirt', 'Fashion', 0.01)]
durable sneakers -> [('Sneakers', 'Fashion', 0.481), ('Keyboard', 'Electronics', 0.092), ('Milk 1L', 'Grocery', 0.072)]
gentle face wash -> [('Face Wash', 'Beauty', 0.737), ('Mouse', 'Electronics', 0.062), ('Perfume', 'Beauty', 0.062)]
top-1 accuracy for exact product-name queries: 1.0
top-3 category hit rate for vague category queries: 1.0


In [ ]:
DATA = "data/"
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

p = pd.read_csv(DATA + "products.csv")
V = TfidfVectorizer()
T = V.fit_transform(p["product"].str.lower() + " " + p.description)
question = "Which product is good for cooking and what does it cost?"
s = cosine_similarity(V.transform([question]), T).ravel()
top = np.argsort(-s)[:2]
context = "\n".join(f"[{i + 1}] {p['product'][j]} (category {p.category[j]}, list price {p.list_price[j]:.0f}): {p.description[j]}" for i, j in enumerate(top))
prompt = ("Answer using ONLY the passages. If the answer is not in them, say 'I do not know'. Cite passage numbers.\n\n"
          f"Passages:\n{context}\n\nQuestion: {question}\nAnswer:")
print(prompt)

Answer using ONLY the passages. If the answer is not in them, say 'I do not know'. Cite passage numbers.

Passages:
[1] T-Shirt (category Fashion, list price 320): compact t-shirt for fashion shoppers, light and organic, good value
[2] Sneakers (category Fashion, list price 680): organic sneakers for fashion shoppers, premium and gentle, good value

Question: Which product is good for cooking and what does it cost?
Answer:


## 7.4 Text-to-SQL: letting people ask the database

An LLM can translate "How many orders were returned in Mumbai?" into SQL, given the table schema. The danger is not that the SQL is wrong sometimes (you can review that), but that it is *executed* automatically: a wrong, slow or destructive query on a production database. The engineering pattern is therefore: schema in the prompt, query out, then a **guard** that allows only read-only statements on approved tables, executed on a read-only connection with a row limit.

In [ ]:
DATA = "data/"
import sqlite3, re

SCHEMA = "orders(order_id, customer_id, order_date, city, channel, category, product, quantity, unit_price, discount, status, delivery_minutes)"
PROMPT = f"You write SQLite. Tables: {SCHEMA}. Return one SELECT statement only. Question: {{q}}"

ALLOWED_TABLES = {"orders", "customers", "products"}
def guard(sql):
    s = sql.strip().rstrip(";")
    if ";" in s:
        return False, "multiple statements"
    if not re.match(r"(?is)^(select|with)\b", s):
        return False, "only SELECT/WITH allowed"
    if re.search(r"(?i)\b(insert|update|delete|drop|alter|create|attach|pragma|replace)\b", s):
        return False, "write keyword found"
    tables = set(re.findall(r"(?i)\b(?:from|join)\s+([a-z_][a-z0-9_]*)", s))
    if not tables <= ALLOWED_TABLES:
        return False, f"table not allowed: {tables - ALLOWED_TABLES}"
    return True, "ok"

con = sqlite3.connect("file:" + DATA + "shopkart.db?mode=ro", uri=True)       # read-only connection
pretend_llm_outputs = {
    "How many orders were returned?": "SELECT COUNT(*) FROM orders WHERE status = 'returned'",
    "Delete all cancelled orders": "DELETE FROM orders WHERE status = 'cancelled'",
    "Show me the employees table": "SELECT * FROM employees",
    "Two things at once": "SELECT 1; DROP TABLE orders",
}
for q, sql in pretend_llm_outputs.items():
    ok, why = guard(sql)
    result = con.execute(sql + " LIMIT 5").fetchall() if ok else None
    print(f"{q!r:36} -> {why:28} {result if ok else ''}")
print(PROMPT.format(q="How many orders were returned?")[:100] + "...")

'How many orders were returned?'     -> ok                           [(2976,)]
'Delete all cancelled orders'        -> only SELECT/WITH allowed     
'Show me the employees table'        -> table not allowed: {'employees'} 
'Two things at once'                 -> multiple statements          
You write SQLite. Tables: orders(order_id, customer_id, order_date, city, channel, category, product...


## 7.5 Recommending products: co-purchase similarity

Before language models, recommenders used behaviour. Build a customer-by-product matrix of purchase counts; two products are similar when the same customers buy both; similarity is the cosine of their columns. No text is needed.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

o, c, p = prep.load_clean(DATA)
M = o.pivot_table(index="customer_id", columns="product", values="order_id", aggfunc="count", fill_value=0)
S = cosine_similarity(M.T.values)
names = list(M.columns)
i = names.index("Earbuds")
order = np.argsort(-S[i])[1:4]
print("matrix:", M.shape, "| Earbuds co-purchase neighbours:", [(names[j], round(float(S[i, j]), 3)) for j in order])
print("spread of similarities between all product pairs:", round(float(np.min(S[np.triu_indices(len(S), 1)])), 3), "to", round(float(np.max(S[np.triu_indices(len(S), 1)])), 3))

matrix: (3215, 32) | Earbuds co-purchase neighbours: [('Noodles', 0.29), ('Rice 5kg', 0.264), ('Milk 1L', 0.264)]
spread of similarities between all product pairs: 0.158 to 0.393


## 7.6 What LLMs add, and when not to use them

## Chapter recap

## Practice set

Problem 7.7.1 Name the four steps of one training iteration of a neural network.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ways/ch07.html